# Route XML Filter
Objective: Filter the complete Bus Routes of Berlin accoarding to a csv containing the desired routes \
Author: Sven Vorgheim \
Disclaimer: The creation of this file was aided by ChatGPT

In [25]:
# Imports
import pandas as pd
from lxml import etree
import math
import os

Assumes the use of "berlin_bus_rou.xml" as given in the BeST-Scenario \
Requieres a "depot_line_type.csv" containing a coloumn "Line" with the desired Lines as string

In [30]:
# Read the csv, select coloumn
specified_routes = pd.read_csv("./depot_line_type.csv",sep=",")
specified = set(specified_routes["line"])
# Parse xml file
tree = etree.parse("../../sumo/berlin_bus.rou.xml")
root = tree.getroot()
# xml anchor contains line as the first part of the anchor
# removes any undesired routes and flows
for route in root.xpath("./route"):
    anchor = route.get("id").split("_", 1)[0]
    if anchor not in specified:
        root.remove(route)

for flow in root.xpath("./flow"):
    anchor = flow.get("id").split("_", 1)[0]
    if anchor not in specified:
        root.remove(flow)

# write result to file as intermediated chache
tree.write(
    "filtered_flows_routes.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)

Calculate additonal information from routes and flows \
Number of simultaneous busses is calculated from route duration and flow period \
nr_of_buses = route.duration/flow.period \
nr_of_repetitions = (flow_end-flow_begin)/duration \
Each Bus on the line must depart $n*period$ after n =1 

Define deadheading routes by Adding departure and arrival areas? Sumo Taz

In [27]:
# Helper Function time in seconds from midnight
def parse_time(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

In [31]:
tree = etree.parse("./filtered_flows_routes.xml")
root = tree.getroot()

rows = []

for i, route in enumerate(root.xpath("./route")):
    anchor = route.get("id")
    stops = route.findall("stop")
    if stops:
        start_stop_id = stops[0].get("busStop")
        end_stop_id = stops[-1].get("busStop")
    for flow in root.xpath("./flow"):
        if flow.get("route") == anchor:
            period = float(flow.get("period"))
            duration = float(route.findall("stop")[-1].get("until"))
            nr_of_buses = math.ceil(int(duration) / int(period))
            flow_end = parse_time(flow.get("end"))
            flow_begin = parse_time(flow.get("begin"))
            # print(flow.get("begin"),flow_begin,flow.get("end"),flow_end)
            nr_of_repetitions = (flow_end-flow_begin)/duration
            nr_of_trips_pd = (flow_end-flow_begin)/period
            # Add the repeat attribute to the route
            # route.set("repeat", str(int(nr_of_repetitions)))
            rows.append({
                "route": anchor,
                "start_stop_id": start_stop_id,
                "end_stop_id": end_stop_id,
                "flow_end": flow_end,
                "flow_begin": flow_begin,
                "period": period,
                "duration": duration,
                "nr_of_buses": nr_of_buses,
                "nr_of_repetitions": int(nr_of_repetitions),
                 "nr_of_trips_pd": int(nr_of_trips_pd)
            })
            root.remove(flow)

# write final result to file
tree.write(
    "cicero_mueller_routes.rou.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)
os.remove("./filtered_flows_routes.xml")

route_calculations = pd.DataFrame(rows)
print(route_calculations)
# Total result of simultaneous buses
print(sum(route_calculations["nr_of_buses"]))
print(sum(route_calculations["nr_of_buses"]*route_calculations["nr_of_repetitions"]))
route_calculations.to_csv("route_calculations.csv", index=False)

        route                                      start_stop_id  \
0    101_9633                                  agg_9633_0_9633_1   
1    101_9646                agg_9633_48_11946_43_11949_0_9646_0   
2    109_9722                                  agg_9722_0_9722_1   
3    109_9724  agg_9627_0_9735_0_10804_0_11658_0_11946_0_1197...   
4    110_9735  agg_9627_0_9735_0_10804_0_11658_0_11946_0_1197...   
..        ...                                                ...   
76  X83_12561                               agg_12570_35_12561_0   
77  X83_12570                        agg_12115_0_12558_0_12570_0   
78  143_12751                                            12751_0   
79  M43_12763                       agg_12788_0_12763_0_12771_36   
80  M43_12771                               agg_12763_36_12771_0   

                                          end_stop_id  flow_end  flow_begin  \
0                 agg_9633_48_11946_43_11949_0_9646_0     85800       15120   
1                        

Created merged csv containing information on Lines

In [32]:
route_df = pd.read_csv("./route_calculations.csv")
line_df = pd.read_csv("./depot_line_type.csv")

# Extract the line name from the route ID
route_df["line"] = route_df["route"].str.split("_").str[0]

# Join on the Line column
merged = route_df.merge(line_df, on="line", how="left")

merged.to_csv("merged_routes.csv", index=False)
os.remove("./route_calculations.csv")
print(merged.head())

      route                                      start_stop_id  \
0  101_9633                                  agg_9633_0_9633_1   
1  101_9646                agg_9633_48_11946_43_11949_0_9646_0   
2  109_9722                                  agg_9722_0_9722_1   
3  109_9724  agg_9627_0_9735_0_10804_0_11658_0_11946_0_1197...   
4  110_9735  agg_9627_0_9735_0_10804_0_11658_0_11946_0_1197...   

                                         end_stop_id  flow_end  flow_begin  \
0                agg_9633_48_11946_43_11949_0_9646_0     85800       15120   
1                                            9646_49     85740        1740   
2  agg_11643_41_11697_35_10630_17_12477_32_10808_...     86160       19860   
3                                            9724_18     86040       19200   
4                                            9735_25     85800       18540   

   period  duration  nr_of_buses  nr_of_repetitions  nr_of_trips_pd line  \
0   600.0    3860.0            7                 18       